In [1]:
!pip install delta-spark --quiet

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

builder = SparkSession.builder \
    .appName("SupplyChainDatabricksSim") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark + Delta session ready:", spark.version)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 788.6 kB/s eta 0:00:00
Spark + Delta session ready: 4.0.3


In [2]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import udf, col, datediff, current_date, when
from pyspark.sql.types import StringType as SparkStringType
from datetime import datetime

In [3]:
schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("supplier_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("order_date", StringType(), True),
    StructField("delivery_date", StringType(), True)
])

In [4]:
# load csv
spark_df = spark.read.csv(
    '/content/orders_01.csv',
    header=True,
    schema=schema
)

In [5]:
# normalize mixed date formats
def normalize_date(date_str):
    if date_str is None or date_str.strip() == "":
        return None
    formats_to_try = ["%Y-%m-%d", "%m/%d/%Y", "%Y/%m/%d"]
    for fmt in formats_to_try:
        try:
            return datetime.strptime(date_str.strip(), fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return None

normalize_date_udf = udf(normalize_date, SparkStringType())

In [6]:
spark_df = spark_df.withColumn("order_date_clean", normalize_date_udf(col("order_date")))
spark_df = spark_df.withColumn("delivery_date_clean", normalize_date_udf(col("delivery_date")))
spark_df = spark_df.withColumn("order_date_parsed", col("order_date_clean").cast("date"))
spark_df = spark_df.withColumn("delivery_date_parsed", col("delivery_date_clean").cast("date"))


In [7]:
# calculate delay_days and is_delayed
spark_df = spark_df.withColumn(
    "delay_days",
    when(
        col("delivery_date_parsed").isNotNull(),
        datediff(current_date(), col("delivery_date_parsed"))
    ).otherwise(
        datediff(current_date(), col("order_date_parsed"))
    )
)

spark_df = spark_df.withColumn(
    "is_delayed",
    when(col("delay_days") > 0, 1).otherwise(0)
)

In [8]:
# drop rows with missing supplier_id (critical field)
spark_df_clean = spark_df.filter(col("supplier_id").isNotNull())

In [9]:
print("Row count after cleaning:", spark_df_clean.count())
spark_df_clean.select("order_id", "supplier_id", "product_id", "quantity",
                        "order_date_parsed", "delivery_date_parsed",
                        "delay_days", "is_delayed").show()

Row count after cleaning: 13
+--------+-----------+----------+--------+-----------------+--------------------+----------+----------+
|order_id|supplier_id|product_id|quantity|order_date_parsed|delivery_date_parsed|delay_days|is_delayed|
+--------+-----------+----------+--------+-----------------+--------------------+----------+----------+
|       1|          1|         2|      50|       2025-06-01|          2025-06-06|       386|         1|
|       2|          2|         5|      30|       2025-06-05|          2025-06-08|       384|         1|
|       3|          3|         1|     100|       2025-06-07|          2025-06-17|       375|         1|
|       4|          4|         6|      20|       2025-06-10|                NULL|       382|         1|
|       5|          5|         3|      75|       2025-06-11|          2025-06-15|       377|         1|
|       6|          1|         7|      15|       2025-06-12|          2025-06-17|       375|         1|
|       7|          2|         4|  

In [10]:
# save the cleaned dataframe as delta format
spark_df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/delta/cleaned_orders")

print("Saved as Delta table at /content/delta/cleaned_orders")

# read it back to verify
delta_check = spark.read.format("delta").load("/content/delta/cleaned_orders")
print("Row count from Delta table:", delta_check.count())
delta_check.show()

Saved as Delta table at /content/delta/cleaned_orders
Row count from Delta table: 13
+--------+-----------+----------+--------+----------+-------------+----------------+-------------------+-----------------+--------------------+----------+----------+
|order_id|supplier_id|product_id|quantity|order_date|delivery_date|order_date_clean|delivery_date_clean|order_date_parsed|delivery_date_parsed|delay_days|is_delayed|
+--------+-----------+----------+--------+----------+-------------+----------------+-------------------+-----------------+--------------------+----------+----------+
|       1|          1|         2|      50|2025-06-01|   2025-06-06|      2025-06-01|         2025-06-06|       2025-06-01|          2025-06-06|       386|         1|
|       2|          2|         5|      30|2025-06-05|   06/08/2025|      2025-06-05|         2025-06-08|       2025-06-05|          2025-06-08|       384|         1|
|       3|          3|         1|     100|2025-06-07|   2025-06-17|      2025-06-07| 

In [11]:
delta_check.createOrReplaceTempView("orders_view")


In [12]:
print("--- Delayed vs On-Time Orders ---")
spark.sql("""
    SELECT is_delayed, COUNT(*) as order_count
    FROM orders_view
    GROUP BY is_delayed
""").show()

--- Delayed vs On-Time Orders ---
+----------+-----------+
|is_delayed|order_count|
+----------+-----------+
|         1|         13|
+----------+-----------+



In [13]:
print("--- Average Delay Days Per Supplier ---")
spark.sql("""
    SELECT supplier_id, ROUND(AVG(delay_days), 2) as avg_delay_days
    FROM orders_view
    GROUP BY supplier_id
    ORDER BY avg_delay_days DESC
""").show()

--- Average Delay Days Per Supplier ---
+-----------+--------------+
|supplier_id|avg_delay_days|
+-----------+--------------+
|          4|         379.0|
|          2|         377.0|
|          1|        376.67|
|          5|         373.5|
|          3|        371.33|
+-----------+--------------+



In [14]:
print("--- Top 3 Most Delayed Orders ---")
spark.sql("""
    SELECT order_id, supplier_id, delay_days
    FROM orders_view
    WHERE is_delayed = 1
    ORDER BY delay_days DESC
    LIMIT 3
""").show()

--- Top 3 Most Delayed Orders ---
+--------+-----------+----------+
|order_id|supplier_id|delay_days|
+--------+-----------+----------+
|       1|          1|       386|
|       2|          2|       384|
|       4|          4|       382|
+--------+-----------+----------+



In [15]:
from pyspark.sql.functions import sum as spark_sum

print("--- Total Quantity Per Supplier (PySpark syntax) ---")
delta_check.groupBy("supplier_id") \
    .agg(spark_sum("quantity").alias("total_quantity")) \
    .orderBy("total_quantity", ascending=False) \
    .show()

--- Total Quantity Per Supplier (PySpark syntax) ---
+-----------+--------------+
|supplier_id|total_quantity|
+-----------+--------------+
|          2|           285|
|          3|           195|
|          4|           110|
|          5|           100|
|          1|            65|
+-----------+--------------+

